In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Any, Dict, List, Tuple
import gurobipy as gp
from gurobipy import GRB
import time

DG_CAPACITY_THRESHOLDS = [60]

ENERGY_DROUGHT_SUMMARY_COLUMNS = [
    "PRLI_Total_kWh",
    "PRLI_Normalized",
    "PRLI_to_TotalDemand",
    "PRLI_PositiveHours",
    "PRLI_PositiveHourShare",
    "PRLI_AveragePositiveResidual_kW",
    "PRLI_MaxPositiveResidual_kW",
    "PRLI_to_PeakDemand",
]

ENERGY_DROUGHT_MONTHLY_COLUMNS = [
    "Month",
    "PRLI_kWh",
    "PRLI_Share",
    "PositiveHours",
    "PositiveHourShare",
    "MaxResidual_kW",
    "MeanPositiveResidual_kW",
]

DG_CONSECUTIVE_COLUMNS = [
    "Threshold_pct",
    "Threshold_kW",
    "MaxConsecutiveHours",
    "EventCount",
    "AvgEventsPerYear",
    "DroughtSignal",
]

# 1. File paths and data loading ------------------------------------------------
# Use repo-friendly relative paths so the notebook can be shared on GitHub.
DATA_FILE_NAME = "input_data_45 El Hierro.xlsx"
DATA_CANDIDATES = [
    Path(DATA_FILE_NAME),
    Path("Data") / DATA_FILE_NAME,
    Path("..") / "Data" / DATA_FILE_NAME,
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        f"Could not find {DATA_FILE_NAME}. Please place it in the notebook folder, Data/, or ../Data/."
    )
OUTPUT_DIR = DATA_PATH.parent / "planning_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def _max_consecutive_hours(values: np.ndarray, threshold: float) -> int:
    if values.size == 0:
        return 0
    above = values >= threshold
    if not np.any(above):
        return 0
    max_streak = 0
    current = 0
    for flag in above:
        if flag:
            current += 1
            if current > max_streak:
                max_streak = current
        else:
            current = 0
    return max_streak


def _count_events(values: np.ndarray, threshold: float) -> int:
    if values.size == 0:
        return 0
    above = values >= threshold
    event_count = 0
    in_event = False
    for flag in above:
        if flag:
            if not in_event:
                event_count += 1
                in_event = True
        else:
            in_event = False
    return event_count


def load_processed_dataset(path: Path) -> pd.DataFrame:
    # Load the processed hourly wind, PV, and demand dataset.
    suffix = path.suffix.lower()
    if suffix == ".csv":
        df = pd.read_csv(path)
    elif suffix in {".xlsx", ".xls"}:
        df = pd.read_excel(path)
    else:
        raise ValueError(f"Unsupported file format: {path}")
    required_cols = {"Year", "HourOfYear", "Timestamp", "CF_Wind", "CF_PV", "Load"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    df = df.copy()
    df["Year"] = df["Year"].astype(int)
    df["HourOfYear"] = df["HourOfYear"].astype(int)
    df["Timestamp"] = pd.to_datetime(df["Timestamp"])
    df["CF_Wind"] = pd.to_numeric(df["CF_Wind"], errors="coerce")
    df["CF_PV"] = pd.to_numeric(df["CF_PV"], errors="coerce")
    df["Load"] = pd.to_numeric(df["Load"], errors="coerce")
    df = df.sort_values(["Year", "HourOfYear"]).reset_index(drop=True)
    df["YearIndex"] = df["Year"] - df["Year"].min() + 1
    return df


def prepare_dataframe(rows: List[Dict[str, Any]]) -> pd.DataFrame:
    df_rows = pd.DataFrame(rows)
    if not df_rows.empty:
        df_rows = df_rows.sort_values(["YearIndex", "CalendarYear"]).reset_index(drop=True)
    return df_rows


# Load the processed multi-year dataset and build the year index mapping.

dataset = load_processed_dataset(DATA_PATH)
year_lookup_df = dataset[["YearIndex", "Year"]].drop_duplicates().sort_values("Year")
year_lookup_df.rename(columns={"Year": "CalendarYear"}, inplace=True)
year_lookup_series = year_lookup_df.set_index("YearIndex")["CalendarYear"]
print("Available year mapping (YearIndex -> CalendarYear):")
print(year_lookup_df.to_string(index=False))

config = {
    "year_range": (1, 45),  # (start_index, end_index) inclusive, 1-based
    "curtailment_rates": [0,5,10,15,20,25,30],  # Always use iteration mode; a single-value list is allowed, e.g. [0] or [20]
    "output_directory": OUTPUT_DIR,
    "mip_gap": 0.01,
    "time_limit_seconds": 240,
    "verbosity": 1,
    "load_multiplier": 1.0,   # 1.0=baseline level, 0.8=-20%, 1.2=+20%
    "load_shape_factor": 1.0, # 1.0=baseline shape, <1 flatter, >1 peakier while preserving annual energy
}

BASE_PARAMS_TEMPLATE = {
    "E_u_min": 0,
    "E_u_max_ub": 225,  # Upper bound for E_u_max decision variable (MWh)
    "P_g_min": 0,
    "P_g_max": 11300,  # Fixed generation power capacity (kW)
    "P_p_min": 0,
    "P_p_max": 6000,  # Fixed pumping power capacity (kW)
    "eta_pump": 0.8,
    "eta_gen": 0.85,
    "Eff_DG": 0.28,     # kWh produced per liter of diesel    L/kWh
    "CO2_DG_operation": 2.65,
    "CO2_PV_LCA": 40,
    "CO2_WT_LCA": 20,
    "CO2_PHS_LCA": 10,
    "Lifetime": 20,
    "Pvcost": 1500,
    "PV_OM": 20,  # PV O&M cost ($/kW-yr) 
    "salvage_rate": 0.20,
    "WTcost": 1700,
    "WT_OM": 30,  # WT O&M cost ($/kW-yr)
    "DGcost": 250,
    "DG_OM": 30,  # DG O&M cost ($/kW-yr)
    "Discountrate": 0.05,
    "FuelCost": 1.5,
    "FuelInflationRate": 0.02,
    "OMInflationRate": 0.02,
    "PV_lifetime": 20,
    "WT_lifetime": 20,
    "PHS_lifetime": 60,
    "DG_lifetime": 10,
    "DG_salvage_rate": 0.10,
    "PHSPowerCost": 1612,  # USD/kW for power capacity (generation + pumping)
    "PHS_Power_OM": 20,  # PHS power O&M cost ($/kW-yr) for (P_g_max + P_p_max)
    "ReservoirCost": 83000,  # USD/MWh for energy storage capacity
    "AllowedLOL": 0,  # Allowed loss-of-load ratio
    "EmissionTreshold": 600,
    "MaxCurtailmentRate": 0,
}

STATUS_MAP = {
    GRB.OPTIMAL: "Optimal",
    GRB.SUBOPTIMAL: "Suboptimal",
    GRB.TIME_LIMIT: "TimeLimit",
    GRB.INFEASIBLE: "Infeasible",
    GRB.INF_OR_UNBD: "InfOrUnbounded",
    GRB.UNBOUNDED: "Unbounded",
    GRB.ITERATION_LIMIT: "IterationLimit",
}

ECONOMIC_COLUMNS = [
    "InitialCost_USD",
    "TotalOMCost_USD",
    "ReplacementCost_USD",
    "SalvageValue_USD",
    "NetPresentCost_USD",
    "AnnualizedCost_USD",
    "LCOE_USD_per_kWh",
    "FuelConsumption_Liters_per_yr",
]

TECHNICAL_COLUMNS = [
    "TotalDemand_kWh",
    "ServedLoad_kWh",
    "UnservedEnergy_kWh",
    "LPSP",
    "PV_Curtailment_Rate",
    "WT_Curtailment_Rate",
    "VRES_Curtailment_Rate",
    "Curtailment_to_Load_Rate",
    "TotalCurtailment_kWh",
    "DG_Generation_kWh",
    "PV_Generation_kWh",
    "WT_Generation_kWh",
    "PHS_Generation_kWh",
    "PHS_Pumping_kWh",
    "Renewable_Share",
    "PeakDemand_kW",
    "PV_CapacityFactor",
    "WT_CapacityFactor",
    "DG_CapacityFactor",
    "PHS_CapacityFactor",
]

CARBON_COLUMNS = [
    "TotalEmissions_kgCO2",
    "EmissionsIntensity_gCO2_per_kWh",
    "Diesel_Operation_kgCO2",
    "PV_LCA_kgCO2",
    "Wind_LCA_kgCO2",
    "PHS_LCA_kgCO2",
]

CAPACITY_COLUMNS = [
    "PV_kW",
    "WT_kW",
    "DG_kW",
    "PHS_Power_Gen_kW",
    "PHS_Power_Pump_kW",
    "PHS_EnergyCapacity_MWh",
]


def build_params(df_year: pd.DataFrame, template: Dict[str, Any]) -> Dict[str, Any]:
    params = template.copy()
    params["SampleHours"] = len(df_year)
    params["M_curtail"] = max(3 * float(df_year["Demand"].max()), 1.0)
    # O&M costs are now directly specified in $/kW-yr, no calculation needed
    params["RealDiscountRate"] = (1 + params["Discountrate"]) / (1 + params["OMInflationRate"]) - 1
    return params


def apply_load_adjustments(df_year: pd.DataFrame, config_dict: Dict[str, Any]) -> pd.DataFrame:
    df = df_year.copy()
    load_multiplier = float(config_dict.get("load_multiplier", 1.0))
    load_shape_factor = float(config_dict.get("load_shape_factor", 1.0))
    if load_multiplier <= 0:
        raise ValueError("load_multiplier must be positive.")
    if load_shape_factor <= 0:
        raise ValueError("load_shape_factor must be positive.")

    adjusted_load = df["Load"].astype(float).to_numpy(dtype=float) * load_multiplier

    # Energy-preserving shape adjustment: L'_t = mu + kappa * (L_t - mu)
    if not np.isclose(load_shape_factor, 1.0):
        mean_load = float(adjusted_load.mean())
        adjusted_load = mean_load + load_shape_factor * (adjusted_load - mean_load)
        if np.any(adjusted_load < 0):
            raise ValueError("load_shape_factor produced negative hourly load values. Please use a milder shape adjustment.")

    df["Load"] = adjusted_load
    return df


def _format_decimal_tag(value: float) -> str:
    rounded = round(float(value), 4)
    if np.isclose(rounded, round(rounded)):
        return str(int(round(rounded)))
    return f"{rounded:.4f}".rstrip("0").rstrip(".").replace(".", "p")


def build_year_range_tag(config_dict: Dict[str, Any]) -> str:
    start_idx, end_idx = config_dict["year_range"]
    return f"Y{start_idx}-{end_idx}"


def build_curtailment_tag(curtailment_rates: List[float]) -> str:
    rates = sorted(float(rate) for rate in curtailment_rates)
    if not rates:
        return "CurtNA"
    if len(rates) == 1:
        return f"Curt{_format_decimal_tag(rates[0])}pct"
    return f"CurtIter_{_format_decimal_tag(rates[0])}to{_format_decimal_tag(rates[-1])}pct_{len(rates)}lvl"


def build_lpsp_tag(params_dict: Dict[str, Any]) -> str:
    allowed_lol_pct = float(params_dict.get("AllowedLOL", 0.0)) * 100.0
    return f"LPSP{_format_decimal_tag(allowed_lol_pct)}pct"


def build_storage_tag(params_dict: Dict[str, Any]) -> str:
    return f"E{_format_decimal_tag(params_dict.get('E_u_max_ub', np.nan))}MWh"


def build_load_scenario_tag(config_dict: Dict[str, Any]) -> str:
    load_multiplier = float(config_dict.get("load_multiplier", 1.0))
    load_shape_factor = float(config_dict.get("load_shape_factor", 1.0))
    multiplier_tag = f"LM{int(round(load_multiplier * 100)):03d}"
    shape_tag = f"LS{int(round(load_shape_factor * 100)):03d}"
    if np.isclose(load_multiplier, 1.0) and np.isclose(load_shape_factor, 1.0):
        return f"LoadBase_{multiplier_tag}_{shape_tag}"
    if not np.isclose(load_multiplier, 1.0) and np.isclose(load_shape_factor, 1.0):
        return f"LoadLevel_{multiplier_tag}_{shape_tag}"
    if np.isclose(load_multiplier, 1.0) and not np.isclose(load_shape_factor, 1.0):
        return f"LoadShape_{multiplier_tag}_{shape_tag}"
    return f"LoadMix_{multiplier_tag}_{shape_tag}"


def build_output_filename(
    config_dict: Dict[str, Any],
    params_dict: Dict[str, Any],
    curtailment_rates: List[float],
) -> Path:
    filename = "_".join([
        "capacity_planning",
        build_year_range_tag(config_dict),
        build_curtailment_tag(curtailment_rates),
        build_lpsp_tag(params_dict),
        build_storage_tag(params_dict),
        build_load_scenario_tag(config_dict),
    ]) + ".xlsx"
    return config_dict["output_directory"] / filename


def build_run_config_table(
    config_dict: Dict[str, Any],
    params_dict: Dict[str, Any],
    curtailment_rates: List[float],
    data_path: Path,
) -> pd.DataFrame:
    start_idx, end_idx = config_dict["year_range"]
    rows = [
        {"Item": "DataFile", "Value": data_path.name},
        {"Item": "YearRangeStart", "Value": start_idx},
        {"Item": "YearRangeEnd", "Value": end_idx},
        {"Item": "CurtailmentRates_pct", "Value": ", ".join(_format_decimal_tag(rate) for rate in curtailment_rates)},
        {"Item": "CurtailmentTag", "Value": build_curtailment_tag(curtailment_rates)},
        {"Item": "AllowedLOL", "Value": params_dict.get("AllowedLOL")},
        {"Item": "LPSPTag", "Value": build_lpsp_tag(params_dict)},
        {"Item": "StorageUpperBound_MWh", "Value": params_dict.get("E_u_max_ub")},
        {"Item": "StorageTag", "Value": build_storage_tag(params_dict)},
        {"Item": "LoadMultiplier", "Value": config_dict.get("load_multiplier", 1.0)},
        {"Item": "LoadShapeFactor", "Value": config_dict.get("load_shape_factor", 1.0)},
        {"Item": "LoadScenarioTag", "Value": build_load_scenario_tag(config_dict)},
        {"Item": "MIPGap", "Value": config_dict.get("mip_gap")},
        {"Item": "TimeLimitSeconds", "Value": config_dict.get("time_limit_seconds")},
        {"Item": "YearRangeTag", "Value": build_year_range_tag(config_dict)},
        {"Item": "ExportTimestamp", "Value": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")},
    ]
    return pd.DataFrame(rows)


def solve_capacity_planning(year_index: int, calendar_year: int, df_year: pd.DataFrame, config_dict: Dict[str, Any]) -> Dict[str, Any]:
    result: Dict[str, Any] = {
        "year_index": year_index,
        "calendar_year": calendar_year,
        "status_code": None,
        "status": None,
        "objective_value": np.nan,
        "objective_bound": np.nan,
        "runtime_seconds": np.nan,
        "gap": np.nan,
        "solution_found": False,
        "economic": None,
        "technical": None,
        "carbon": None,
        "capacity": None,
        "energy_drought": None,
        "notes": None,
    }

    df = df_year.copy()
    df.rename(columns={"Load": "Demand"}, inplace=True)
    df["Demand"] = df["Demand"].astype(float)
    df["CF_PV"] = df["CF_PV"].astype(float)
    df["CF_Wind"] = df["CF_Wind"].astype(float)
    df = df.reset_index(drop=True)

    demand = df["Demand"].to_numpy(dtype=float)
    cf_pv = df["CF_PV"].to_numpy(dtype=float)
    cf_wind = df["CF_Wind"].to_numpy(dtype=float)

    total_demand = float(demand.sum())
    peak_demand = float(demand.max())
    if total_demand <= 0:
        result["notes"] = "Total demand for the selected year is zero or negative."
        return result

    params = build_params(df, BASE_PARAMS_TEMPLATE)
    T = params["SampleHours"]

    try:
        m = gp.Model(f"SolarWindDieselPHS_{calendar_year}")
        if config_dict.get("verbosity", 1) == 0:
            m.Params.OutputFlag = 0
        m.Params.MIPGap = config_dict.get("mip_gap", 0.01)
        m.Params.TimeLimit = config_dict.get("time_limit_seconds", 150)

        # Decision variables ----------------------------------------------------
        PVSZ = m.addVar(lb=0, ub=1_000_000, name="PVSZ")
        WTSZ = m.addVar(lb=0, ub=1_000_000, name="WTSZ")
        DGSZ = m.addVar(lb=0, ub=2 * peak_demand, name="DGSZ")
        E_u_max_var = m.addVar(lb=params["E_u_max_ub"], ub=params["E_u_max_ub"], name="E_u_max")

        PV_Load = m.addVars(T, lb=0, name="PV_Load")
        WT_Load = m.addVars(T, lb=0, name="WT_Load")
        DG_Load = m.addVars(T, lb=0, name="DG_Load")
        PV_curtailed = m.addVars(T, lb=0, name="PV_curtailed")
        WT_curtailed = m.addVars(T, lb=0, name="WT_curtailed")
        loadloss = m.addVars(T, lb=0, name="loadloss")

        PV_PHS = m.addVars(T, lb=0, name="PV_PHS")
        WT_PHS = m.addVars(T, lb=0, name="WT_PHS")
        PHS_g = m.addVars(T, lb=0, name="PHS_g")
        PHS_p = m.addVars(T, lb=0, name="PHS_p")
        E_u = m.addVars(T + 1, lb=params["E_u_min"], name="E_u")

        I_p = m.addVars(T, vtype=GRB.BINARY, name="I_p")
        I_g = m.addVars(T, vtype=GRB.BINARY, name="I_g")

        m.addConstr(E_u[0] == 0.8 * E_u_max_var, name="E_u_initial")
        m.addConstr(E_u[T] == 0.8 * E_u_max_var, name="E_u_final")
        m.addConstrs((E_u[t] <= E_u_max_var for t in range(T + 1)), name="E_u_capacity")
        m.addConstrs((E_u[t + 1] == E_u[t] - PHS_g[t] / params["eta_gen"] / 1000 + PHS_p[t] * params["eta_pump"] / 1000 for t in range(T)), name="E_u_balance")

        m.addConstrs(((I_g[t] == 0) >> (PHS_g[t] == 0) for t in range(T)), name="gen_off")
        m.addConstrs(((I_p[t] == 0) >> (PHS_p[t] == 0) for t in range(T)), name="pump_off")
        m.addConstrs(((I_g[t] == 1) >> (PHS_g[t] >= params["P_g_min"]) for t in range(T)), name="gen_min_on")
        m.addConstrs(((I_g[t] == 1) >> (PHS_g[t] <= params["P_g_max"]) for t in range(T)), name="gen_max_on")
        m.addConstrs(((I_p[t] == 1) >> (PHS_p[t] >= params["P_p_min"]) for t in range(T)), name="pump_min_on")
        m.addConstrs(((I_p[t] == 1) >> (PHS_p[t] <= params["P_p_max"]) for t in range(T)), name="pump_max_on")
        m.addConstrs((I_g[t] + I_p[t] <= 1 for t in range(T)), name="PHS_state")

        # Curtailment is allowed during pumping, but not during PHS generation.
        m.addConstrs((PV_curtailed[t] + WT_curtailed[t] <= params["M_curtail"] * (1 - I_g[t]) for t in range(T)), name="No_PHS_gen_when_curtail")

        m.addConstrs((PV_Load[t] + PV_PHS[t] + PV_curtailed[t] == cf_pv[t] * PVSZ for t in range(T)), name="PV_gen")
        m.addConstrs((WT_Load[t] + WT_PHS[t] + WT_curtailed[t] == cf_wind[t] * WTSZ for t in range(T)), name="WT_gen")
        m.addConstrs((PHS_p[t] == PV_PHS[t] + WT_PHS[t] for t in range(T)), name="PHS_pumping_source")
        m.addConstrs((DG_Load[t] <= DGSZ for t in range(T)), name="DG_capacity")

        m.addConstrs((PV_Load[t] + WT_Load[t] + DG_Load[t] + PHS_g[t] + loadloss[t] == demand[t] for t in range(T)), name="Load_balance")

        m.addConstr(loadloss.sum() <= params["AllowedLOL"] * total_demand, name="LOL_Constraint")

        total_renewable_available = gp.quicksum((cf_pv[t] * PVSZ + cf_wind[t] * WTSZ) for t in range(T))
        total_curtailed_expr = PV_curtailed.sum() + WT_curtailed.sum()
        m.addConstr(total_curtailed_expr <= params["MaxCurtailmentRate"] * total_renewable_available, name="Curtailment_Constraint")

        served_load_expr = total_demand - loadloss.sum()
        emissions_numerator_expr = (
            DG_Load.sum() * params["Eff_DG"] * params["CO2_DG_operation"]
            + (PV_Load.sum() + PV_PHS.sum() + PV_curtailed.sum()) * params["CO2_PV_LCA"] / 1000
            + (WT_Load.sum() + WT_PHS.sum() + WT_curtailed.sum()) * params["CO2_WT_LCA"] / 1000
            + (PHS_g.sum() + PHS_p.sum()) * params["CO2_PHS_LCA"] / 1000
        )
        m.addConstr(emissions_numerator_expr * 1000 <= params["EmissionTreshold"] * served_load_expr, name="Emissions_Constraint")

        # Objective -------------------------------------------------------------
        PHS_power_cost = params["PHSPowerCost"] * (params["P_g_max"] + params["P_p_max"])
        Initialcost_expr = (
            params["Pvcost"] * PVSZ
            + params["WTcost"] * WTSZ
            + params["DGcost"] * DGSZ
            + PHS_power_cost
            + params["ReservoirCost"] * E_u_max_var
        )
        annual_fuel_consumption_expr = DG_Load.sum() * params["Eff_DG"]
        phs_total_power_kw = params["P_g_max"] + params["P_p_max"]
        yearly_capacity_om_expr = (
            params["PV_OM"] * PVSZ
            + params["WT_OM"] * WTSZ
            + params["DG_OM"] * DGSZ
            + params["PHS_Power_OM"] * phs_total_power_kw
        )

        OMcost_expr = gp.LinExpr()
        for y in range(1, params["Lifetime"] + 1):
            discount_factor = 1 / ((1 + params["RealDiscountRate"]) ** y)
            OMcost_expr += (yearly_capacity_om_expr + annual_fuel_consumption_expr * params["FuelCost"]) * discount_factor

        Replacement_cost_expr = gp.LinExpr()
        for replacement_year in range(params["DG_lifetime"], params["Lifetime"], params["DG_lifetime"]):
            replacement_discount = 1 / ((1 + params["RealDiscountRate"]) ** replacement_year)
            Replacement_cost_expr += DGSZ * params["DGcost"] * (1 - params["DG_salvage_rate"]) * replacement_discount

        TotalOMcost_expr = OMcost_expr + Replacement_cost_expr
        final_discount = 1 / ((1 + params["RealDiscountRate"]) ** params["Lifetime"])
        pv_salvage_rate = max(0, params["PV_lifetime"] - params["Lifetime"]) / params["PV_lifetime"] if params["PV_lifetime"] > 0 else 0
        wt_salvage_rate = max(0, params["WT_lifetime"] - params["Lifetime"]) / params["WT_lifetime"] if params["WT_lifetime"] > 0 else 0
        dg_remaining_years = params["DG_lifetime"] - (params["Lifetime"] % params["DG_lifetime"])
        dg_salvage_rate = dg_remaining_years / params["DG_lifetime"] if dg_remaining_years > 0 else 0
        phs_salvage_rate = max(0, params["PHS_lifetime"] - params["Lifetime"]) / params["PHS_lifetime"] if params["PHS_lifetime"] > 0 else 0

        pv_salvage_expr = pv_salvage_rate * params["Pvcost"] * PVSZ * final_discount
        wt_salvage_expr = wt_salvage_rate * params["WTcost"] * WTSZ * final_discount
        dg_salvage_expr = dg_salvage_rate * params["DGcost"] * DGSZ * final_discount
        phs_power_salvage_expr = phs_salvage_rate * PHS_power_cost * final_discount
        phs_reservoir_salvage_expr = phs_salvage_rate * params["ReservoirCost"] * E_u_max_var * final_discount

        SalvageValue_expr = pv_salvage_expr + wt_salvage_expr + dg_salvage_expr + phs_power_salvage_expr + phs_reservoir_salvage_expr
        NetPresentCost_expr = Initialcost_expr + TotalOMcost_expr - SalvageValue_expr

        CRF = (
            params["RealDiscountRate"] * (1 + params["RealDiscountRate"]) ** params["Lifetime"]
        ) / (
            (1 + params["RealDiscountRate"]) ** params["Lifetime"] - 1
        )
        served_load_target = total_demand * (1 - params["AllowedLOL"])
        LCOE_expr = (NetPresentCost_expr * CRF) / served_load_target
        m.setObjective(LCOE_expr, GRB.MINIMIZE)

        # Solve ----------------------------------------------------------------
        solve_start = time.time()
        m.optimize()
        solve_end = time.time()
        elapsed = solve_end - solve_start

        result["status_code"] = m.Status
        result["status"] = STATUS_MAP.get(m.Status, f"Status_{m.Status}")
        result["runtime_seconds"] = float(elapsed)
        result["objective_bound"] = float(getattr(m, "ObjBound", np.nan))

        if m.SolCount == 0:
            result["notes"] = "No feasible solution returned by solver."
            return result

        result["solution_found"] = True
        result["objective_value"] = float(m.ObjVal)
        try:
            result["gap"] = float(m.MIPGap)
        except AttributeError:
            result["gap"] = np.nan

        # Extract solution metrics --------------------------------------------
        PVSZ_sol = float(PVSZ.X)
        WTSZ_sol = float(WTSZ.X)
        DGSZ_sol = float(DGSZ.X)
        E_u_max_sol = float(E_u_max_var.X)

        DG_Load_sol_sum = float(sum(DG_Load[t].X for t in range(T)))
        PV_total_generation = float(sum(PV_Load[t].X + PV_PHS[t].X for t in range(T)))
        WT_total_generation = float(sum(WT_Load[t].X + WT_PHS[t].X for t in range(T)))
        PHS_total_generation = float(sum(PHS_g[t].X for t in range(T)))
        PHS_total_pumping = float(sum(PHS_p[t].X for t in range(T)))
        PV_curtailed_total = float(sum(PV_curtailed[t].X for t in range(T)))
        WT_curtailed_total = float(sum(WT_curtailed[t].X for t in range(T)))
        load_loss_sum = float(sum(loadloss[t].X for t in range(T)))

        pv_load_arr = np.array([PV_Load[t].X for t in range(T)], dtype=float)
        wt_load_arr = np.array([WT_Load[t].X for t in range(T)], dtype=float)
        dg_load_arr = np.array([DG_Load[t].X for t in range(T)], dtype=float)
        phs_gen_arr = np.array([PHS_g[t].X for t in range(T)], dtype=float)
        load_loss_arr = np.array([loadloss[t].X for t in range(T)], dtype=float)

        served_load = total_demand - load_loss_sum
        available_pv = float(cf_pv.sum() * PVSZ_sol)
        available_wind = float(cf_wind.sum() * WTSZ_sol)
        total_renewable_available_value = available_pv + available_wind
        total_curtailed_actual = PV_curtailed_total + WT_curtailed_total
        renewable_generation_served = PV_total_generation + WT_total_generation

        pv_curtail_rate = PV_curtailed_total / available_pv if available_pv > 0 else 0.0
        wt_curtail_rate = WT_curtailed_total / available_wind if available_wind > 0 else 0.0
        total_curtail_rate = total_curtailed_actual / total_renewable_available_value if total_renewable_available_value > 0 else 0.0
        curtail_to_load_rate = total_curtailed_actual / total_demand if total_demand > 0 else 0.0
        renewable_share = renewable_generation_served / served_load if served_load > 0 else np.nan

        hours = float(T) if T > 0 else np.nan
        pv_capacity_factor = PV_total_generation / (PVSZ_sol * hours) if PVSZ_sol > 0 and hours else np.nan
        wt_capacity_factor = WT_total_generation / (WTSZ_sol * hours) if WTSZ_sol > 0 and hours else np.nan
        dg_capacity_factor = DG_Load_sol_sum / (DGSZ_sol * hours) if DGSZ_sol > 0 and hours else np.nan
        phs_capacity_factor = PHS_total_generation / (params["P_g_max"] * hours) if params["P_g_max"] > 0 and hours else np.nan

        annual_fuel_consumption = DG_Load_sol_sum * params["Eff_DG"]
        annual_fuel_cost = annual_fuel_consumption * params["FuelCost"]
        PHS_power_cost_value = params["PHSPowerCost"] * (params["P_g_max"] + params["P_p_max"])
        phs_total_power_kw_value = params["P_g_max"] + params["P_p_max"]
        yearly_capacity_om_value = (
            params["PV_OM"] * PVSZ_sol
            + params["WT_OM"] * WTSZ_sol
            + params["DG_OM"] * DGSZ_sol
            + params["PHS_Power_OM"] * phs_total_power_kw_value
        )

        OMcost_value = 0.0
        for y in range(1, params["Lifetime"] + 1):
            discount_factor = 1 / ((1 + params["RealDiscountRate"]) ** y)
            OMcost_value += (yearly_capacity_om_value + annual_fuel_cost) * discount_factor

        replacement_cost_value = 0.0
        for replacement_year in range(params["DG_lifetime"], params["Lifetime"], params["DG_lifetime"]):
            replacement_discount = 1 / ((1 + params["RealDiscountRate"]) ** replacement_year)
            replacement_cost_value += DGSZ_sol * params["DGcost"] * (1 - params["DG_salvage_rate"]) * replacement_discount

        total_om_cost_value = OMcost_value + replacement_cost_value

        pv_salvage_rate_val = pv_salvage_rate
        wt_salvage_rate_val = wt_salvage_rate
        dg_salvage_rate_val = dg_salvage_rate
        phs_salvage_rate_val = phs_salvage_rate
        salvage_value = (
            pv_salvage_rate_val * params["Pvcost"] * PVSZ_sol * final_discount
            + wt_salvage_rate_val * params["WTcost"] * WTSZ_sol * final_discount
            + dg_salvage_rate_val * params["DGcost"] * DGSZ_sol * final_discount
            + phs_salvage_rate_val * PHS_power_cost_value * final_discount
            + phs_salvage_rate_val * params["ReservoirCost"] * E_u_max_sol * final_discount
        )

        initial_cost_value = (
            params["Pvcost"] * PVSZ_sol
            + params["WTcost"] * WTSZ_sol
            + params["DGcost"] * DGSZ_sol
            + PHS_power_cost_value
            + params["ReservoirCost"] * E_u_max_sol
        )
        net_present_cost_value = initial_cost_value + total_om_cost_value - salvage_value
        CRF_value = CRF
        annualized_cost = net_present_cost_value * CRF_value
        lcoe_value = annualized_cost / served_load if served_load > 0 else np.nan
        # Post-process emissions: diesel operation + VRES LCA (including curtailment) + PHS LCA.
        emissions_sources_kg = {
            "Diesel_Operation": DG_Load_sol_sum * params["Eff_DG"] * params["CO2_DG_operation"],
            "PV_LCA": (PV_total_generation + PV_curtailed_total) * params["CO2_PV_LCA"] / 1000,
            "Wind_LCA": (WT_total_generation + WT_curtailed_total) * params["CO2_WT_LCA"] / 1000,
            "PHS_LCA": (PHS_total_generation + PHS_total_pumping) * params["CO2_PHS_LCA"] / 1000,
        }
        total_emissions_kg = float(sum(emissions_sources_kg.values()))
        emissions_intensity = total_emissions_kg * 1000 / served_load if served_load > 0 else np.nan

        residual_after_vres_storage = demand - (pv_load_arr + wt_load_arr + phs_gen_arr)
        positive_residual = np.clip(residual_after_vres_storage, 0, None)
        prli_total = float(positive_residual.sum())
        positive_hours = int(np.count_nonzero(positive_residual > 0))
        positive_hour_share = positive_hours / T if T > 0 else np.nan
        prli_normalized = prli_total / (peak_demand * T) if peak_demand > 0 and T > 0 else np.nan
        prli_to_total_demand = prli_total / total_demand if total_demand > 0 else np.nan
        prli_to_peak_demand = prli_total / peak_demand if peak_demand > 0 else np.nan
        avg_positive_residual = prli_total / positive_hours if positive_hours > 0 else np.nan
        max_positive_residual = float(positive_residual.max()) if positive_residual.size > 0 else np.nan

        monthly_records: List[Dict[str, Any]] = []
        monthly_df = pd.DataFrame({
            "Timestamp": df["Timestamp"],
            "PositiveResidual_kW": positive_residual,
        })
        if not monthly_df.empty:
            monthly_df["Month"] = monthly_df["Timestamp"].dt.month
            monthly_group = monthly_df.groupby("Month", sort=True)
            for month, group in monthly_group:
                prli_month = float(group["PositiveResidual_kW"].sum())
                positive_hours_month = int(np.count_nonzero(group["PositiveResidual_kW"] > 0))
                prli_share_month = prli_month / prli_total if prli_total > 0 else 0.0
                positive_hour_share_month = (
                    positive_hours_month / positive_hours if positive_hours > 0 else 0.0
                )
                positive_values = group.loc[group["PositiveResidual_kW"] > 0, "PositiveResidual_kW"]
                mean_positive = float(positive_values.mean()) if not positive_values.empty else 0.0
                max_positive = float(group["PositiveResidual_kW"].max()) if not group.empty else 0.0
                monthly_records.append({
                    "Month": int(month),
                    "PRLI_kWh": prli_month,
                    "PRLI_Share": float(prli_share_month),
                    "PositiveHours": positive_hours_month,
                    "PositiveHourShare": float(positive_hour_share_month),
                    "MaxResidual_kW": max_positive,
                    "MeanPositiveResidual_kW": mean_positive,
                })

        dg_consecutive_records: List[Dict[str, Any]] = []
        for threshold_pct in DG_CAPACITY_THRESHOLDS:
            threshold_kw = float(DGSZ_sol * threshold_pct / 100.0)
            if DGSZ_sol > 0 and T > 0:
                max_hours = _max_consecutive_hours(dg_load_arr, threshold_kw)
                event_count = _count_events(dg_load_arr, threshold_kw)
                avg_events_per_year_value = float(event_count * (8760.0 / T))
                drought_signal = (dg_load_arr >= threshold_kw).astype(int).tolist()
            else:
                max_hours = 0
                event_count = 0
                avg_events_per_year_value = np.nan
                drought_signal = [0 for _ in range(T)] if T > 0 else []
            dg_consecutive_records.append({
                "Threshold_pct": threshold_pct,
                "Threshold_kW": threshold_kw,
                "MaxConsecutiveHours": int(max_hours),
                "EventCount": int(event_count),
                "AvgEventsPerYear": avg_events_per_year_value,
                "DroughtSignal": drought_signal,
            })

        economic_metrics = {
            "InitialCost_USD": initial_cost_value,
            "TotalOMCost_USD": total_om_cost_value,
            "ReplacementCost_USD": replacement_cost_value,
            "SalvageValue_USD": salvage_value,
            "NetPresentCost_USD": net_present_cost_value,
            "AnnualizedCost_USD": annualized_cost,
            "LCOE_USD_per_kWh": lcoe_value,
            "FuelConsumption_Liters_per_yr": annual_fuel_consumption,
        }

        technical_metrics = {
            "TotalDemand_kWh": total_demand,
            "ServedLoad_kWh": served_load,
            "UnservedEnergy_kWh": load_loss_sum,
            "LPSP": load_loss_sum / total_demand if total_demand > 0 else np.nan,
            "PV_Curtailment_Rate": pv_curtail_rate,
            "WT_Curtailment_Rate": wt_curtail_rate,
            "VRES_Curtailment_Rate": total_curtail_rate,
            "Curtailment_to_Load_Rate": curtail_to_load_rate,
            "TotalCurtailment_kWh": total_curtailed_actual,
            "DG_Generation_kWh": DG_Load_sol_sum,
            "PV_Generation_kWh": PV_total_generation,
            "WT_Generation_kWh": WT_total_generation,
            "PHS_Generation_kWh": PHS_total_generation,
            "PHS_Pumping_kWh": PHS_total_pumping,
            "Renewable_Share": renewable_share,
            "PeakDemand_kW": peak_demand,
            "PV_CapacityFactor": pv_capacity_factor,
            "WT_CapacityFactor": wt_capacity_factor,
            "DG_CapacityFactor": dg_capacity_factor,
            "PHS_CapacityFactor": phs_capacity_factor,
        }

        carbon_metrics = {
            "TotalEmissions_kgCO2": total_emissions_kg,
            "EmissionsIntensity_gCO2_per_kWh": emissions_intensity,
            "Diesel_Operation_kgCO2": emissions_sources_kg["Diesel_Operation"],
            "PV_LCA_kgCO2": emissions_sources_kg["PV_LCA"],
            "Wind_LCA_kgCO2": emissions_sources_kg["Wind_LCA"],
            "PHS_LCA_kgCO2": emissions_sources_kg["PHS_LCA"],
        }

        capacity_metrics = {
            "PV_kW": PVSZ_sol,
            "WT_kW": WTSZ_sol,
            "DG_kW": DGSZ_sol,
            "PHS_Power_Gen_kW": params["P_g_max"],
            "PHS_Power_Pump_kW": params["P_p_max"],
            "PHS_EnergyCapacity_MWh": E_u_max_sol,
        }

        result["economic"] = economic_metrics
        result["technical"] = technical_metrics
        result["carbon"] = carbon_metrics
        result["capacity"] = capacity_metrics
        result["energy_drought"] = {
            "prli_total_kwh": prli_total,
            "prli_normalized": prli_normalized,
            "prli_to_total_demand": prli_to_total_demand,
            "prli_positive_hours": positive_hours,
            "prli_positive_hour_share": positive_hour_share,
            "average_positive_residual": avg_positive_residual,
            "max_positive_residual": max_positive_residual,
            "prli_to_peak_demand": prli_to_peak_demand,
            "monthly_records": monthly_records,
            "dg_consecutive": dg_consecutive_records,
        }
        result["notes"] = (
            f"LCOE={lcoe_value:.4f} USD/kWh | PV={PVSZ_sol:.2f} kW | WT={WTSZ_sol:.2f} kW | "
            f"DG={DGSZ_sol:.2f} kW | PHS_E={E_u_max_sol:.2f} MWh | Runtime={elapsed:.1f} s"
        )
    except gp.GurobiError as exc:
        result["status"] = "ERROR"
        result["notes"] = str(exc)
        print(f"YearIndex {year_index} (Year {calendar_year}) failed with error: {exc}")

    return result


start_idx, end_idx = config["year_range"]
if start_idx < 1 or end_idx > len(year_lookup_series) or start_idx > end_idx:
    raise ValueError("Invalid year_range configuration. Ensure 1 <= start <= end <= total years.")

selected_indices = list(range(start_idx, end_idx + 1))
selected_pairs: List[Tuple[int, int]] = [(idx, int(year_lookup_series.loc[idx])) for idx in selected_indices]

# Build curtailment-rate sequence
curtailment_rates = config.get("curtailment_rates", [0])
if not isinstance(curtailment_rates, (list, tuple)) or len(curtailment_rates) == 0:
    raise ValueError("curtailment_rates must be a non-empty list or tuple.")
curtailment_rates = [float(rate) for rate in curtailment_rates]
print(f"\n=== Curtailment iteration mode: {len(curtailment_rates)} rate levels ===")
print(f"Curtailment-rate sequence: {curtailment_rates}%")
print(f"Load scenario: {build_load_scenario_tag(config)}\n")

# Outer loop: curtailment-rate iteration
all_results: Dict[float, List[Dict[str, Any]]] = {}
for curtail_rate in curtailment_rates:
    print(f"\n{'='*80}")
    print(f"Start solving for curtailment rate = {curtail_rate}%")
    print(f"{'='*80}\n")
    
    # Update the maximum curtailment rate in the parameter template
    BASE_PARAMS_TEMPLATE["MaxCurtailmentRate"] = curtail_rate / 100.0
    
    results: List[Dict[str, Any]] = []
    for year_index, calendar_year in selected_pairs:
        print(f"  Processing year: YearIndex={year_index}, CalendarYear={calendar_year}, CurtailmentRate={curtail_rate}%")
        df_year = dataset.loc[dataset["Year"] == calendar_year, ["Timestamp", "CF_PV", "CF_Wind", "Load"]].reset_index(drop=True)
        df_year = apply_load_adjustments(df_year, config)
        result = solve_capacity_planning(year_index, calendar_year, df_year, config)
        # Add curtailment-rate metadata to the result record
        result["curtailment_rate_pct"] = curtail_rate
        results.append(result)
    
    all_results[curtail_rate] = results
    print(f"\nCurtailment rate {curtail_rate}% completed.\n")

# Organize result tables by curtailment rate
curtailment_results_dict: Dict[float, Dict[str, pd.DataFrame]] = {}

for curtail_rate, results in all_results.items():
    economic_rows: List[Dict[str, Any]] = []
    technical_rows: List[Dict[str, Any]] = []
    carbon_rows: List[Dict[str, Any]] = []
    capacity_rows: List[Dict[str, Any]] = []
    meta_rows: List[Dict[str, Any]] = []
    energy_drought_rows: List[Dict[str, Any]] = []
    energy_drought_monthly_rows: List[Dict[str, Any]] = []
    dg_consecutive_rows: List[Dict[str, Any]] = []

    for res in results:
        base_info = {
            "YearIndex": res["year_index"], 
            "CalendarYear": res["calendar_year"],
            "CurtailmentRate_pct": res.get("curtailment_rate_pct", curtail_rate),
            "LoadScenarioTag": build_load_scenario_tag(config),
            "YearRangeTag": build_year_range_tag(config),
            "LPSPTag": build_lpsp_tag(BASE_PARAMS_TEMPLATE),
            "StorageTag": build_storage_tag(BASE_PARAMS_TEMPLATE),
        }

        econ_row = base_info.copy()
        for col in ECONOMIC_COLUMNS:
            econ_row[col] = np.nan
        if res["solution_found"] and res["economic"]:
            econ_row.update(res["economic"])
        economic_rows.append(econ_row)

        tech_row = base_info.copy()
        for col in TECHNICAL_COLUMNS:
            tech_row[col] = np.nan
        if res["solution_found"] and res["technical"]:
            tech_row.update(res["technical"])
        technical_rows.append(tech_row)

        carbon_row = base_info.copy()
        for col in CARBON_COLUMNS:
            carbon_row[col] = np.nan
        if res["solution_found"] and res["carbon"]:
            carbon_row.update(res["carbon"])
        carbon_rows.append(carbon_row)

        capacity_row = base_info.copy()
        for col in CAPACITY_COLUMNS:
            capacity_row[col] = np.nan
        if res["solution_found"] and res["capacity"]:
            capacity_row.update(res["capacity"])
        capacity_rows.append(capacity_row)

        energy_row = base_info.copy()
        for col in ENERGY_DROUGHT_SUMMARY_COLUMNS:
            energy_row[col] = np.nan
        if res["solution_found"] and res.get("energy_drought"):
            ed = res["energy_drought"]
            energy_row.update({
                "PRLI_Total_kWh": ed.get("prli_total_kwh", np.nan),
                "PRLI_Normalized": ed.get("prli_normalized", np.nan),
                "PRLI_to_TotalDemand": ed.get("prli_to_total_demand", np.nan),
                "PRLI_PositiveHours": ed.get("prli_positive_hours", np.nan),
                "PRLI_PositiveHourShare": ed.get("prli_positive_hour_share", np.nan),
                "PRLI_AveragePositiveResidual_kW": ed.get("average_positive_residual", np.nan),
                "PRLI_MaxPositiveResidual_kW": ed.get("max_positive_residual", np.nan),
                "PRLI_to_PeakDemand": ed.get("prli_to_peak_demand", np.nan),
            })
            for monthly_record in ed.get("monthly_records", []):
                monthly_row = base_info.copy()
                monthly_row.update(monthly_record)
                energy_drought_monthly_rows.append(monthly_row)
            for dg_record in ed.get("dg_consecutive", []):
                dg_row = base_info.copy()
                dg_row.update(dg_record)
                dg_consecutive_rows.append(dg_row)
        energy_drought_rows.append(energy_row)

        meta_row = base_info.copy()
        meta_row.update({
            "StatusCode": res["status_code"],
            "Status": res["status"],
            "SolutionFound": res["solution_found"],
            "ObjectiveValue": res["objective_value"],
            "ObjectiveBound": res["objective_bound"],
            "Gap": res["gap"],
            "RuntimeSeconds": res["runtime_seconds"],
            "Notes": res["notes"],
        })
        meta_rows.append(meta_row)

    economic_df = prepare_dataframe(economic_rows)
    technical_df = prepare_dataframe(technical_rows)
    carbon_df = prepare_dataframe(carbon_rows)
    capacity_df = prepare_dataframe(capacity_rows)
    energy_drought_df = prepare_dataframe(energy_drought_rows)
    meta_df = prepare_dataframe(meta_rows)
    energy_drought_monthly_df = prepare_dataframe(energy_drought_monthly_rows)
    if not energy_drought_monthly_df.empty:
        energy_drought_monthly_df = energy_drought_monthly_df.sort_values(["CurtailmentRate_pct", "YearIndex", "CalendarYear", "Month"]).reset_index(drop=True)
    dg_consecutive_df = prepare_dataframe(dg_consecutive_rows)
    if not dg_consecutive_df.empty:
        dg_consecutive_df = dg_consecutive_df.sort_values(["CurtailmentRate_pct", "YearIndex", "CalendarYear", "Threshold_pct"]).reset_index(drop=True)

    summary_table = meta_df.copy()
    if not economic_df.empty:
        summary_table = summary_table.merge(
            economic_df[["CurtailmentRate_pct", "YearIndex", "CalendarYear", "LCOE_USD_per_kWh", "NetPresentCost_USD"]],
            on=["CurtailmentRate_pct", "YearIndex", "CalendarYear"],
            how="left",
        )
    if not capacity_df.empty:
        summary_table = summary_table.merge(
            capacity_df[["CurtailmentRate_pct", "YearIndex", "CalendarYear"] + CAPACITY_COLUMNS],
            on=["CurtailmentRate_pct", "YearIndex", "CalendarYear"],
            how="left",
        )
    if not carbon_df.empty:
        summary_table = summary_table.merge(
            carbon_df[["CurtailmentRate_pct", "YearIndex", "CalendarYear", "EmissionsIntensity_gCO2_per_kWh", "TotalEmissions_kgCO2"]],
            on=["CurtailmentRate_pct", "YearIndex", "CalendarYear"],
            how="left",
        )
    if not technical_df.empty:
        summary_table = summary_table.merge(
            technical_df[["CurtailmentRate_pct", "YearIndex", "CalendarYear"] + TECHNICAL_COLUMNS],
            on=["CurtailmentRate_pct", "YearIndex", "CalendarYear"],
            how="left",
        )
    if not energy_drought_df.empty:
        summary_table = summary_table.merge(
            energy_drought_df[["CurtailmentRate_pct", "YearIndex", "CalendarYear"] + ENERGY_DROUGHT_SUMMARY_COLUMNS],
            on=["CurtailmentRate_pct", "YearIndex", "CalendarYear"],
            how="left",
        )

    summary_table = summary_table.sort_values(["CurtailmentRate_pct", "YearIndex", "CalendarYear"]).reset_index(drop=True)
    
    # Store all result tables for this curtailment rate
    curtailment_results_dict[curtail_rate] = {
        "summary": summary_table,
        "economic": economic_df,
        "technical": technical_df,
        "carbon": carbon_df,
        "capacity": capacity_df,
        "energy_drought": energy_drought_df,
        "energy_drought_monthly": energy_drought_monthly_df,
        "dg_consecutive": dg_consecutive_df,
        "meta": meta_df,
    }

# Build cross-rate comparison table (stats for spectrum/band visualization)
comparison_rows = []
for curtail_rate, dfs in curtailment_results_dict.items():
    summary_df = dfs["summary"]
    if not summary_df.empty and summary_df["SolutionFound"].any():
        # Extract key time-series indicators
        lcoe_series = summary_df["LCOE_USD_per_kWh"].dropna()
        emissions_series = summary_df["EmissionsIntensity_gCO2_per_kWh"].dropna()
        curtail_series = summary_df["VRES_Curtailment_Rate"].dropna()
        
        # Compute across-year statistics for this curtailment rate
        avg_row = {
            "CurtailmentRate_pct": curtail_rate,
            "NumYears": len(summary_df),
            
            # LCOE statistics (for spectrum plotting)
            "LCOE_Mean": lcoe_series.mean() if len(lcoe_series) > 0 else np.nan,
            "LCOE_Std": lcoe_series.std() if len(lcoe_series) > 0 else np.nan,
            "LCOE_Min": lcoe_series.min() if len(lcoe_series) > 0 else np.nan,
            "LCOE_Max": lcoe_series.max() if len(lcoe_series) > 0 else np.nan,
            "LCOE_Median": lcoe_series.median() if len(lcoe_series) > 0 else np.nan,
            "LCOE_Q25": lcoe_series.quantile(0.25) if len(lcoe_series) > 0 else np.nan,
            "LCOE_Q75": lcoe_series.quantile(0.75) if len(lcoe_series) > 0 else np.nan,
            "LCOE_CV": (lcoe_series.std() / lcoe_series.mean()) if len(lcoe_series) > 0 and lcoe_series.mean() > 0 else np.nan,
            
            # Emission-intensity statistics (for spectrum plotting)
            "EmissionsIntensity_Mean": emissions_series.mean() if len(emissions_series) > 0 else np.nan,
            "EmissionsIntensity_Std": emissions_series.std() if len(emissions_series) > 0 else np.nan,
            "EmissionsIntensity_Min": emissions_series.min() if len(emissions_series) > 0 else np.nan,
            "EmissionsIntensity_Max": emissions_series.max() if len(emissions_series) > 0 else np.nan,
            "EmissionsIntensity_Median": emissions_series.median() if len(emissions_series) > 0 else np.nan,
            "EmissionsIntensity_Q25": emissions_series.quantile(0.25) if len(emissions_series) > 0 else np.nan,
            "EmissionsIntensity_Q75": emissions_series.quantile(0.75) if len(emissions_series) > 0 else np.nan,
            "EmissionsIntensity_CV": (emissions_series.std() / emissions_series.mean()) if len(emissions_series) > 0 and emissions_series.mean() > 0 else np.nan,
            
            # Actual curtailment-rate statistics (for spectrum plotting)
            "ActualCurtailment_Mean": curtail_series.mean() if len(curtail_series) > 0 else np.nan,
            "ActualCurtailment_Std": curtail_series.std() if len(curtail_series) > 0 else np.nan,
            "ActualCurtailment_Min": curtail_series.min() if len(curtail_series) > 0 else np.nan,
            "ActualCurtailment_Max": curtail_series.max() if len(curtail_series) > 0 else np.nan,
            "ActualCurtailment_Median": curtail_series.median() if len(curtail_series) > 0 else np.nan,
            "ActualCurtailment_Q25": curtail_series.quantile(0.25) if len(curtail_series) > 0 else np.nan,
            "ActualCurtailment_Q75": curtail_series.quantile(0.75) if len(curtail_series) > 0 else np.nan,
            "ActualCurtailment_CV": (curtail_series.std() / curtail_series.mean()) if len(curtail_series) > 0 and curtail_series.mean() > 0 else np.nan,
            
            # Mean values of other key indicators
            "AvgNPC_USD": summary_df["NetPresentCost_USD"].mean(),
            "AvgLPSP": summary_df["LPSP"].mean(),
            "AvgRenewable_Share": summary_df["Renewable_Share"].mean(),
            "AvgPV_kW": summary_df["PV_kW"].mean(),
            "AvgWT_kW": summary_df["WT_kW"].mean(),
            "AvgDG_kW": summary_df["DG_kW"].mean(),
            "AvgPHS_EnergyCapacity_MWh": summary_df["PHS_EnergyCapacity_MWh"].mean(),
        }
        comparison_rows.append(avg_row)

comparison_df = pd.DataFrame(comparison_rows)
if not comparison_df.empty:
    comparison_df = comparison_df.sort_values("CurtailmentRate_pct").reset_index(drop=True)

# Save aggregated results to Excel
output_file = build_output_filename(config, BASE_PARAMS_TEMPLATE, curtailment_rates)
run_config_df = build_run_config_table(config, BASE_PARAMS_TEMPLATE, curtailment_rates, DATA_PATH)

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Write the comparison table across curtailment rates
    if not comparison_df.empty:
        comparison_df.to_excel(writer, sheet_name="CurtailmentComparison", index=False)

    # Write scenario metadata for downstream plotting and batch reads
    run_config_df.to_excel(writer, sheet_name="RunConfig", index=False)
    
    # Create separate sheets for each curtailment rate
    for curtail_rate in sorted(curtailment_results_dict.keys()):
        dfs = curtailment_results_dict[curtail_rate]
        sheet_prefix = f"C{int(curtail_rate)}pct"
        
        # Limit sheet names to Excel's 31-character maximum
        dfs["summary"].to_excel(writer, sheet_name=f"{sheet_prefix}_Summary"[:31], index=False)
        dfs["economic"].to_excel(writer, sheet_name=f"{sheet_prefix}_Economic"[:31], index=False)
        dfs["technical"].to_excel(writer, sheet_name=f"{sheet_prefix}_Technical"[:31], index=False)
        dfs["carbon"].to_excel(writer, sheet_name=f"{sheet_prefix}_Carbon"[:31], index=False)
        dfs["capacity"].to_excel(writer, sheet_name=f"{sheet_prefix}_Capacity"[:31], index=False)
        dfs["energy_drought"].to_excel(writer, sheet_name=f"{sheet_prefix}_ED_Sum"[:31], index=False)
        if not dfs["energy_drought_monthly"].empty:
            dfs["energy_drought_monthly"].to_excel(writer, sheet_name=f"{sheet_prefix}_ED_Mon"[:31], index=False)
        if not dfs["dg_consecutive"].empty:
            dfs["dg_consecutive"].to_excel(writer, sheet_name=f"{sheet_prefix}_DGCon"[:31], index=False)
        dfs["meta"].to_excel(writer, sheet_name=f"{sheet_prefix}_Meta"[:31], index=False)
    
    # Add the year-index mapping table
    year_lookup_df.to_excel(writer, sheet_name="YearIndexMap", index=False)

print(f"\n{'='*80}")
print("All calculations completed.")
print(f"{'='*80}")
print(f"\nResults saved to: {output_file}")
print(f"Curtailment rates evaluated: {curtailment_rates}%")
print(f"Year range: {start_idx} to {end_idx}")
print(f"LPSP setting: {build_lpsp_tag(BASE_PARAMS_TEMPLATE)}")
print(f"Storage setting: {build_storage_tag(BASE_PARAMS_TEMPLATE)}")
print(f"Load scenario: {build_load_scenario_tag(config)}")

if not comparison_df.empty:
    print("\n\nCurtailment comparison summary (mean values across simulated years):")
    print("="*120)
    print(comparison_df.round(4).to_string(index=False))
    print("="*120)

